<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/04_srgan_x3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SRGAN ×3


## 1. Data

In [ ]:
import sys, urllib.request
LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'srgan_models.py', 'srgan_losses.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # drop a stale copy if one is already imported

from sr_utils import *

show_data()        # 2 validation patches + 2 test areas

## 2. Training


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from srgan_models import Generator, Discriminator
from srgan_losses import GeneratorLoss

N_TRAIN, EPOCHS, BATCH = 16, 1, 4
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
to_t = lambda a: torch.from_numpy(np.stack(a).transpose(0, 3, 1, 2)).float() / 255
loader = DataLoader(TensorDataset(to_t(lo), to_t(hi)), batch_size=BATCH, shuffle=True)

netG, netD = Generator(3).to(dev).train(), Discriminator().to(dev).train()
optG, optD = torch.optim.Adam(netG.parameters()), torch.optim.Adam(netD.parameters())
crit = GeneratorLoss().to(dev)

for ep in range(1, EPOCHS + 1):
    gl = dl = dx = dgz = 0.0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        fake = netG(x)                                     # 1) generator
        g_loss = crit(netD(fake).mean(), fake, y)
        optG.zero_grad(); g_loss.backward(); optG.step()

        real_out, fake_out = netD(y).mean(), netD(fake.detach()).mean()
        d_loss = 1 - real_out + fake_out                   # 2) discriminator
        optD.zero_grad(); d_loss.backward(); optD.step()

        gl += g_loss.item(); dl += d_loss.item()
        dx += real_out.item(); dgz += fake_out.item()
    k = len(loader)
    print(f'epoch {ep}/{EPOCHS}  Loss_G {gl/k:.4f}  Loss_D {dl/k:.4f}  '
          f'D(x) {dx/k:.3f}  D(G(z)) {dgz/k:.3f}')

## 3. Results

In [ ]:
MODEL = f'{BASE}/models/04_srgan_x3'

from srgan_models import load_srgan

net = load_srgan(fetch(f'{MODEL}/checkpoints/srgan_g_x3.pth', 'srgan_x3.pth'))

@torch.no_grad()
def upscale(lr):
    t = torch.from_numpy(lr.transpose(2, 0, 1)).float()[None].to(next(net.parameters()).device) / 255
    return (net(t).clamp(0, 1)[0].cpu().numpy().transpose(1, 2, 0) * 255).round().astype('uint8')

show_results(upscale, 'SRGAN', center=[(67, 370), (313, 36)])

## 4. Final test

A real Sentinel-2 capture, with no ground truth.  Judged by eye.


In [ ]:
show_test(upscale, 'SRGAN', center=(800, 800), size=70)